# TP : Détection de Fissures (Cameroun & CFD)
Ce notebook implémente un modèle complet de vision par ordinateur pour détecter et évaluer la sévérité des fissures sur les routes. Il respecte les 4 étapes de l'architecture demandée :
1. Segmentation & 2. Architecture Attention U-Net
3. Gestion du déséquilibre (Augmentation & Loss)
4. Métriques et Sévérité

--- INITIALISATION ET IMPORTATIONS ---

In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from google.colab import drive

# ==========================================
# CHEMINS DES DONNÉES (Dataset Kaggle)
# ==========================================
# S'assurer d'avoir téléchargé et extrait le dataset dans le dossier 'data'
chemin_images = './data/images/'
chemin_masques = './data/masks/'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

Mounted at /content/drive
Appareil utilisé : cuda


--- PREPARATION DES DONNEES (DATALOADER) ---

## Étape 1 & 3: DataLoader, Augmentation et Oversampling
**Objectif :** Charger les données pour la classification *Pixel-wise*.
**Gestion du déséquilibre :**
- `ElasticTransform` (Déformation élastique) et `CoarseDropout` (Cutout)
- Oversampling : Le script force le choix de patches $256 \times 256$ contenant des fissures.

In [3]:
# --- PREPARATION DES DONNEES (DATALOADER CORRIGÉ) ---
import os
import cv2
import numpy as np
import random
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class FissureDataset(Dataset):
    def __init__(self, image_dir, mask_dir, patch_size=256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
        self.patch_size = patch_size

        self.transform = A.Compose([
            A.PadIfNeeded(min_height=patch_size, min_width=patch_size, border_mode=cv2.BORDER_CONSTANT, fill=0),
            A.RandomCrop(width=patch_size, height=patch_size),
            A.ElasticTransform(p=0.5, alpha=120, sigma=6.0),
            A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 32), hole_width_range=(1, 32), p=0.5),
            A.HorizontalFlip(p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.images) * 5

    def __getitem__(self, idx):
        img_name = self.images[idx % len(self.images)]
        img_path = os.path.join(self.image_dir, img_name)

        nom_base = img_name.rsplit('.', 1)[0]
        mask_path = os.path.join(self.mask_dir, nom_base + '.png')

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.float32)
        else:
            # S'il n'y a pas de masque du tout, on crée un masque vide (Que du négatif)
            mask = np.zeros(image.shape[:2], dtype=np.float32)

        # 30% du temps, on cherche VOLONTAIREMENT un patch sans fissure (ex: herbe, route saine)
        veut_negatif = random.random() < 0.30

        for _ in range(10): # On essaie 10 fois de trouver le bon patch
            augmented = self.transform(image=image, mask=mask)
            somme_pixels = augmented['mask'].sum()

            if veut_negatif and somme_pixels == 0:
                # On a trouvé un patch 100% propre (voiture, herbe, etc.), on le garde !
                return augmented['image'], augmented['mask'].unsqueeze(0)

            elif not veut_negatif and somme_pixels > (self.patch_size * self.patch_size * 0.01):
                # On a trouvé un patch avec une fissure, on le garde !
                return augmented['image'], augmented['mask'].unsqueeze(0)

        # Si après 10 essais on n'a pas trouvé exactement ce qu'on voulait, on renvoie ce qu'on a
        return augmented['image'], augmented['mask'].unsqueeze(0)

# On recrée le loader
train_dataset = FissureDataset(CHEMIN_IMAGES, CHEMIN_MASQUES)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

--- MODELE ATTENTION U-NET COMPLETE ---

## Étape 2 : Architecture Attention U-Net
Implémentation stricte :
- **Encoder :** 4 blocs (Conv - BatchNorm - ReLU + Maxpooling).
- **Decoder :** 4 blocs avec upsampling.
- **Skip Connections :** Filtrées par des *Attention Gates*.
- **Output :** Couche 1x1 avec activation Sigmoïde.

In [4]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, kernel_size=1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, kernel_size=1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, kernel_size=1), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        psi = F.relu(self.W_g(g) + self.W_x(x))
        return x * self.psi(psi)

class AttentionUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.e1 = ConvBlock(3, 64)
        self.e2 = ConvBlock(64, 128)
        self.e3 = ConvBlock(128, 256)
        self.e4 = ConvBlock(256, 512)
        self.b = ConvBlock(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.att4 = AttentionGate(512, 512, 256)
        self.d4 = ConvBlock(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.att3 = AttentionGate(256, 256, 128)
        self.d3 = ConvBlock(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.att2 = AttentionGate(128, 128, 64)
        self.d2 = ConvBlock(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.att1 = AttentionGate(64, 64, 32)
        self.d1 = ConvBlock(128, 64)

        self.out = nn.Sequential(nn.Conv2d(64, 1, 1), nn.Sigmoid())

    def forward(self, x):
        x1 = self.e1(x); p1 = self.pool(x1)
        x2 = self.e2(p1); p2 = self.pool(x2)
        x3 = self.e3(p2); p3 = self.pool(x3)
        x4 = self.e4(p3); p4 = self.pool(x4)
        b = self.b(p4)

        d4 = self.up4(b); x4 = self.att4(d4, x4); d4 = torch.cat((x4, d4), dim=1); d4 = self.d4(d4)
        d3 = self.up3(d4); x3 = self.att3(d3, x3); d3 = torch.cat((x3, d3), dim=1); d3 = self.d3(d3)
        d2 = self.up2(d3); x2 = self.att2(d2, x2); d2 = torch.cat((x2, d2), dim=1); d2 = self.d2(d2)
        d1 = self.up1(d2); x1 = self.att1(d1, x1); d1 = torch.cat((x1, d1), dim=1); d1 = self.d1(d1)

        return self.out(d1)

model = AttentionUNet().to(device)

--- LOSS ET ENTRAINEMENT ---

## Étape 3 (Suite) : Fonction de Perte et Entraînement
Combinaison mathématique pour combattre le déséquilibre :
$Loss = 0.7 \times Dice + 0.3 \times BCE$

In [5]:
class DiceBCELoss(nn.Module):
    def __init__(self, weight_dice=0.7, weight_bce=0.3):
        super().__init__()
        self.wd = weight_dice
        self.wb = weight_bce

    def forward(self, pred, target):
        bce = F.binary_cross_entropy(pred, target)
        smooth = 1e-5
        intersection = (pred * target).sum()
        dice_loss = 1 - ((2. * intersection + smooth) / (pred.sum() + target.sum() + smooth))
        return self.wb * bce + self.wd * dice_loss

criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/10 - Loss: 0.8413
Epoch 2/10 - Loss: 0.7990
Epoch 3/10 - Loss: 0.7822
Epoch 4/10 - Loss: 0.7646
Epoch 5/10 - Loss: 0.7555
Epoch 6/10 - Loss: 0.7454
Epoch 7/10 - Loss: 0.7332
Epoch 8/10 - Loss: 0.7269
Epoch 9/10 - Loss: 0.7263
Epoch 10/10 - Loss: 0.7113


 --- SAUVEGARDE DU MODÈLE ENTRAÎNÉ ---µ

In [6]:
chemin_sauvegarde = '/content/attention_unet_fissures.pth'
torch.save(model.state_dict(), chemin_sauvegarde)
print(f"Modèle sauvegardé avec succès sous : {chemin_sauvegarde}")

Modèle sauvegardé avec succès sous : /content/attention_unet_fissures.pth


 --- METRIQUES ET SEVERITE (POST-TRAITEMENT) ---

## Étape 4 : Métriques & Classification de Sévérité
Post-traitement des prédictions :
- Binarisation via **Seuil d'Otsu**.
- Nettoyage via **Morphologie mathématique** (Fermeture).
- Extraction de métriques (Longueur, Largeur, Densité).
- Règles métier pour la classification.

In [7]:
def evaluer_fissure(prediction_masque_brut):
    # 1. Seuil d'Otsu (Convertir probabilités en 0-255 pour OpenCV)
    pred_img = (prediction_masque_brut * 255).astype(np.uint8)
    _, masque_binaire = cv2.threshold(pred_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 2. Morphologie (Fermer les petits trous)
    kernel = np.ones((3,3), np.uint8)
    masque_propre = cv2.morphologyEx(masque_binaire, cv2.MORPH_CLOSE, kernel)

    # 3. Métriques : Densité, Longueur et Largeur (Approximations via contours)
    surface_totale = masque_propre.shape[0] * masque_propre.shape[1]
    pixels_fissure = cv2.countNonZero(masque_propre)
    densite = (pixels_fissure / surface_totale) * 100

    contours, _ = cv2.findContours(masque_propre, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    longueur_max = 0
    largeur_max = 0

    if contours:
        for cnt in contours:
            # Longueur basée sur le périmètre
            longueur = cv2.arcLength(cnt, False) / 2
            if longueur > longueur_max:
                longueur_max = longueur
                # Largeur approximée par l'aire divisée par la longueur
                aire = cv2.contourArea(cnt)
                largeur_max = aire / longueur if longueur > 0 else 0

    # 4. Classification de Sévérité (Règles métier)
    severite = "Saine"
    if densite > 5 or largeur_max > 10:
        severite = "Critique (Intervention urgente)"
    elif densite > 1 or longueur_max > 50:
        severite = "Moyenne (A surveiller)"
    elif pixels_fissure > 0:
        severite = "Légère"

    return {
        "Densité (%)": round(densite, 2),
        "Longueur max (px)": round(longueur_max, 2),
        "Largeur max (px)": round(largeur_max, 2),
        "Sévérité": severite
    }

# Exemple d'utilisation (simulé) :
masque_simule = np.random.rand(256, 256) # Simule la sortie du modèle
print(evaluer_fissure(masque_simule))

{'Densité (%)': 98.46, 'Longueur max (px)': 504.13, 'Largeur max (px)': 128.72, 'Sévérité': 'Critique (Intervention urgente)'}


--- INTERFACE GRADI0 POUR LA PRESENTATION ---

In [8]:
#!pip install gradio -q

import gradio as gr
import cv2
import numpy as np
import torch

# On s'assure que le modèle est en mode "évaluation" (et non entraînement)
model.eval()

def inspecter_route(image_numpy):
    """Fonction appelée quand on clique sur "Submit" dans l'interface"""
    # 1. Redimensionner l'image envoyée à 256x256 (taille attendue par ton modèle)
    img_resized = cv2.resize(image_numpy, (256, 256))

    # 2. Normalisation (même calcul que dans ton dataset)
    img_norm = img_resized.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_norm = (img_norm - mean) / std

    # 3. Préparer pour PyTorch et envoyer sur le GPU
    tensor_img = torch.tensor(img_norm).permute(2, 0, 1).unsqueeze(0).float().to(device)

    # 4. Faire la prédiction avec ton Attention U-Net
    with torch.no_grad():
        prediction = model(tensor_img)
        masque_brut = prediction.squeeze().cpu().numpy() # On récupère le masque 2D

    # 5. Obtenir les métriques avec ta fonction de l'étape 4
    rapport = evaluer_fissure(masque_brut)

    # 6. Créer le visuel Waouh ! (Colorier la fissure en rouge vif sur l'image)
    masque_binaire = (masque_brut > 0.5).astype(np.uint8) * 255
    calque_rouge = np.zeros_like(img_resized)
    calque_rouge[:, :, 0] = masque_binaire # Canal Rouge activé là où il y a la fissure

    # On superpose l'image originale et le calque rouge
    image_resultat = cv2.addWeighted(img_resized, 0.8, calque_rouge, 0.8, 0)

    # 7. Formater le texte pour l'affichage
    texte_final = f"🚨 SÉVÉRITÉ : {rapport['Sévérité']}\n"
    texte_final += f"━" * 30 + "\n"
    texte_final += f"📏 Longueur max : {rapport['Longueur max (px)']} pixels\n"
    texte_final += f"↔️ Largeur max : {rapport['Largeur max (px)']} pixels\n"
    texte_final += f"🕸️ Densité de dégradation : {rapport['Densité (%)']} %"

    return image_resultat, texte_final

# --- CRÉATION DE L'INTERFACE WEB ---
interface = gr.Interface(
    fn=inspecter_route,
    inputs=gr.Image(label="1. Uploadez une image de route (Caméroun ou CFD)"),
    outputs=[
        gr.Image(label="2. Détection Attention U-Net"),
        gr.Textbox(label="3. Rapport d'Inspection Technique", lines=6)
    ],
    title="🚧 Système d'Inspection des Infrastructures par IA",
    description="Modèle Attention U-Net entraîné sur des données mixtes. Détection au pixel près et règles métiers de priorisation.",
    theme="default"
)

# share=True génère un lien public !
interface.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://47c97d6bafe8c2a660.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
